# GRPO su GSM8K con TRL

Questo notebook spiega passo per passo gli oggetti coinvolti nel training GRPO con TRL:
- **GSM8K** — il dataset di problemi matematici
- **`accuracy_reward`** — la reward function di TRL
- **`GRPOTrainer`** — il trainer che orchestra tutto

Obiettivo finale: fare training con esattamente questo snippet:
```python
trainer = GRPOTrainer(model="...", reward_funcs=accuracy_reward, train_dataset=dataset)
trainer.train()
```

In [ ]:
# !pip install -q trl datasets math_verify  # math_verify è richiesto da accuracy_reward

## 1. GSM8K — struttura del dataset

In [1]:
from datasets import load_dataset

raw = load_dataset("openai/gsm8k", "main")
print(raw)
# DatasetDict con split 'train' (7473 esempi) e 'test' (1319)

/work/fis1/RT-DeepRL/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})


In [ ]:
# Ogni esempio ha solo due colonne: question e answer
example = raw["train"][0]
print("COLONNE:", raw["train"].column_names)
print()
print("QUESTION:\n", example["question"])
print()
print("ANSWER (raw):\n", example["answer"])
# La risposta contiene il ragionamento step-by-step e il numero finale dopo '####'

In [ ]:
import re

def extract_number(answer_text: str) -> str | None:
    """Estrae il numero finale dopo #### come stringa."""
    match = re.search(r"####\s*(-?[\d,]+)", answer_text)
    if match:
        return match.group(1).replace(",", "")  # rimuove le virgole (es. 1,000 -> 1000)
    return None

print("Numero estratto:", extract_number(example["answer"]))

## 2. Formattare il dataset per `GRPOTrainer`

`GRPOTrainer` si aspetta due colonne obbligatorie:
| Colonna | Tipo | Descrizione |
|---------|------|-------------|
| `prompt` | `list[dict]` | Chat messages (almeno il turno `user`) |
| `solution` | `str` | La risposta attesa, passata as-is alla reward function |

In [ ]:
SYSTEM = (
    "You are a math assistant. Reason step by step, "
    r"then box your final answer with: \boxed{<number>}"
)

def format_for_grpo(example):
    solution = extract_number(example["answer"])
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM},
            {"role": "user",   "content": example["question"]},
        ],
        "solution": solution,
    }

train_ds = (
    raw["train"]
    .map(format_for_grpo)
    .filter(lambda x: x["solution"] is not None)
    .remove_columns(["question", "answer"])
)

print(train_ds)
print()
print("Esempio formattato:")
print(train_ds[0])

## 3. `accuracy_reward` — come funziona

```python
accuracy_reward(completions, solution, **kwargs) -> list[float | None]
```

- **`completions`**: generazioni del modello — `list[list[dict]]`, ognuna è `[{"role": "assistant", "content": "..."}]`
- **`solution`**: ground truth estratta dal dataset — `list[str]`
- Usa `math_verify` per fare parsing LaTeX/numerico e confronto simbolico
- Ritorna `1.0` (corretto), `0.0` (sbagliato), o `None` (gold non parseable → esempio skippato)

In [ ]:
from trl.rewards import accuracy_reward

# accuracy_reward usa math_verify che cerca \boxed{} nelle completion — NON il formato #### 42
fake_completions = [
    [{"role": "assistant", "content": r"Let me think... The answer is \boxed{42}"}],
    [{"role": "assistant", "content": r"I think the answer is \boxed{100}"}],
    [{"role": "assistant", "content": r"Step 1: ... Therefore, \boxed{42}"}],
]
fake_solutions = ["42", "42", "99"]

rewards = accuracy_reward(fake_completions, fake_solutions)
for comp, sol, r in zip(fake_completions, fake_solutions, rewards):
    content = comp[0]["content"]
    print(f"completion: {content!r}  |  solution: {sol!r}  |  reward: {r}")
# atteso: 1.0, 0.0, 0.0

**Nota**: `accuracy_reward` si aspetta che il gold (`solution`) sia una stringa parseable da `math_verify`. Un numero intero come `"42"` funziona. Se il gold non è parseable, la reward è `None` e il trainer skippa quell'esempio.

## 4. `GRPOTrainer` — il loop completo

**GRPO** (Group Relative Policy Optimization) funziona così:
1. Per ogni prompt, genera **G completions** con il modello corrente
2. Calcola la reward per ognuna con `reward_funcs`
3. Normalizza le reward all'interno del gruppo (media 0, std 1) → *advantage*
4. Fa un passo di policy gradient pesato dall'advantage, con clip KL dalla reference policy

`GRPOTrainer` si aspetta:
- `model`: path/nome HuggingFace o `PreTrainedModel`
- `reward_funcs`: funzione o lista di funzioni con firma `(completions, **dataset_columns) -> list[float]`
- `train_dataset`: dataset con colonne `prompt` e le colonne extra usate dalla reward (qui `solution`)

In [ ]:
import torch
from trl import GRPOTrainer, GRPOConfig

# GRPOConfig eredita da TrainingArguments — tutti i parametri standard HF sono disponibili
training_args = GRPOConfig(
    output_dir="grpo-gsm8k",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,       # G: completions per prompt per calcolare il vantaggio
    max_completion_length=512,
    learning_rate=1e-6,
    logging_steps=10,
    bf16=torch.cuda.is_available(),
)

trainer = GRPOTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    reward_funcs=accuracy_reward,   # il trainer passa 'solution' dal dataset automaticamente
    args=training_args,
    train_dataset=train_ds,
)

print("Trainer pronto. Colonne del dataset:", trainer.train_dataset.column_names)

## 5. Training

In [ ]:
trainer.train()

## Riepilogo: il flusso completo in un blocco

In [ ]:
import re
from datasets import load_dataset
import torch
from trl import GRPOTrainer, GRPOConfig
from trl.rewards import accuracy_reward

def format_gsm8k(example):
    match = re.search(r"####\s*(-?[\d,]+)", example["answer"])
    solution = match.group(1).replace(",", "") if match else None
    return {
        "prompt": [
            {"role": "system", "content": r"Solve the math problem step by step. Box your final answer: \boxed{<number>}"},
            {"role": "user",   "content": example["question"]},
        ],
        "solution": solution,
    }

dataset = (
    load_dataset("openai/gsm8k", "main", split="train")
    .map(format_gsm8k)
    .filter(lambda x: x["solution"] is not None)
    .remove_columns(["question", "answer"])
)

trainer = GRPOTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    reward_funcs=accuracy_reward,
    args=GRPOConfig(output_dir="grpo-gsm8k", num_generations=4, bf16=torch.cuda.is_available()),
    train_dataset=dataset,
)

trainer.train()